In [1]:
#加载环境变量
from dotenv import load_dotenv

load_dotenv()

True


一个完整的Agent至少要包含两个关键的部分:

+ 模型: 是Agent的大脑，负责推理、分析，规划任务步骤
+ 工具: 是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下:

+ 定义工具
+ 初始化模型
+ 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的工具(Tool)，本质就是一个可调用的函数，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息:

+ 工具名
+ 工具的作用
+ 工具需要的参数

## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用:

In [6]:
from langchain.tools import tool

@tool("square_root", description="Calculates the square root of a given number.")
def tool1(x: float) -> float:
    return x ** 0.5 

## 1.2.使用函数名和文档注释描述工具
如果不@tool装饰器没有定义工具名和作用描述,此时:
+ 工具名:默认就是函数名
+ 工具所需的参数:默认就是函数的参数列表
+ 工具作用的描述:默认就是函数的文档注释

In [ ]:
from langchain_core.tools import tool
#通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5


In [ ]:
#定义一个查询天气的tool
from unittest import result

@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


## 1.3.定义Pydantic Model描述参数
在 LangChain 里，Pydantic 主要有三大用途：定义工具的参数、约束大模型的最终结构化输出、定义多Agent之间的状态。

Pydantic 的实质就是一份规范。

如果函数的参数比较多,而且比较复杂,此时建议通过pydantic model来描述参数列表。

In [14]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal

#例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

In [15]:
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp}degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [16]:
square_root.invoke({"x": 467})

21.61018278497431

In [20]:
get_weather.invoke({"location":"杭州","include_forecast":False})

'Current weather in 杭州: 22degrees C'

In [21]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    tools=[square_root, get_weather]
)


# 4. 调用
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="广州接下来是什么天气?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)



Current weather in 广州: 22degrees C
Next 5 days: Sunny广州目前的温度为22摄氏度。未来5天的天气预报预计为晴天。

In [22]:
response = agent.invoke(
{"messages":[HumanMessage(content="467和529的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少?
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_23a7e914b0c64dcc86933cf8)
 Call ID: call_23a7e914b0c64dcc86933cf8
  Args:
    x: 467
  square_root (call_5e9130a529324a3b9df226e7)
 Call ID: call_5e9130a529324a3b9df226e7
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

467和529的平方根分别是：

*   **467** 的平方根约为 **21.61** (精确值为 21.61018278497431)
*   **529** 的平方根为 **23**
None


# 2.预定义Tool（20-LangChain入门-预定义工具Tavily）
LangChain中提供了很多预定义的Tool,方便我们使用。例如:
+ tavily:就是一个用来做web搜索的工具
## 2.1.基本用法
它的使用步骤是这样的:
+ 注册账号,创建API_KEY
+ 配置环境变量:TAVILY_API_KEY
+ 安装依赖:uv add langchain-tavily(或者pip install tavily-python)

In [13]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_tavily import TavilySearch

# 1. 加载环境变量
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 初始化千问模型（你的四件套）
from langchain.chat_models import init_chat_model
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 3. 初始化搜索工具
search_tool = TavilySearch(max_results=3, tavily_api_key=os.getenv("TAVILY_API_KEY"))

# 4. 创建真正的 Agent（把模型和工具绑在一起）
agent = create_agent(
    model=qwen_model,
    tools=[search_tool],
    system_prompt="你是一个聪明的助手，遇到不懂的梗，请使用搜索工具去查，并用简短的话告诉用户结果。"
)


'''
# ========================================================第一种打印
# 5. 让 Agent 去查“蒸蚌”
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是啥梗？")]}
)

print(response["messages"][-1].content)
'''

# ========================================================第二种打印
response = agent.invoke(
    {"messages":[HumanMessage(content="蒸蚌是什么梗?")]},
)
for message in response['messages']:
    message.pretty_print()

"""
# ========================================================第三种打印
import json
search_result = tool.invoke("蒸蚌是啥梗")

# 只提取前3条结果的标题和链接，美化打印
for i, res in enumerate(search_result["results"], 1):
    print(f"{i}. {res['title']}")
    print(f"   链接: {res['url']}\n")
"""



'''
"DeepSeek export chat history JSON parser github"

"AI chat context summarizer tool github"

"LLM chat log to markdown conversion"
'''

================================ Human Message =================================

蒸蚌是什么梗?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_e16886b12ec54e21a469a883)
 Call ID: call_e16886b12ec54e21a469a883
  Args:
    query: 蒸蚌是什么梗
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.sina.cn/news/detail/5247814261015270.html", "title": "#周深 蒸蚌#“蒸蚌”=“真棒”的谐音梗，这次出圈就是“萝卜纸巾猫”二创的番外彩蛋：1. 梗怎么来的 ◦ 起源是抖音博主@超级无敌大开门训练三花猫分辨“萝卜/纸巾”，猫咪靠“察言观色”蒙答案，主人用米老鼠式高音喊“真棒”。 ◦ 视频被搬运到微博后，评论区把“真棒”空耳成“蒸蚌”，配上猫咪 ​_新浪新闻", "content": "发布于 广西\n\n首页\n\n导航\n\n反馈 [...] 汽车 本地生活 旅游 数码 娱乐 影视\n\n财娱乐\n\n25-12-25 18:51 [...] #周深 蒸蚌#  \n“蒸蚌”=“真棒”的谐音梗，这次出圈就是“萝卜纸巾猫”二创的番外彩蛋：  \n1. 梗怎么来的  \n ◦ 起源是抖音博主@超级无敌大开门训练三花猫分辨“萝卜/纸巾”，猫咪靠“察言观色”蒙答案，主人用米老鼠式高音喊“真棒”。  \n ◦ 视频被搬运到微博后，评论区把“真棒”空耳成“蒸蚌”，配上猫咪一脸懵的截图，形成魔性谐音。  \n2. 周深怎么“接

''

### 发散的思维之炼丹（核心是提取与大模型某一聊天框的全部内容或主要内容）

一、你的“双层炼丹”方案提炼（简历/面试绝佳素材）
你提出的这个方案，本质上是一套“AI 对话上下文管理的高可用架构”。我帮你梳理成清晰的结构：

🎯 核心痛点：AI 聊天框存在上下文窗口硬限制。一旦满了或意外卡死，你连发送“炼丹指令”的机会都没有，进度直接丢失。

🛡️ 第一层：手动防御（常规操作）

流程：感觉对话快满 -> 发送《丹方》给 AI -> AI 按固定章节输出“丹” -> 手动复制到新聊天框。

局限：依赖用户主观感受，如果遇到突发死循环或卡死，此层直接失效。

🚀 第二层：自动兜底（极致方案）

流程：利用 DeepSeek 等平台开放的数据导出功能（JSON/MD 格式聊天记录）-> 结合 TavilySearch 检索 GitHub 开源社区寻找现成工具 -> 若无则自己开发 -> 编写 Python 脚本本地解析 JSON -> 调用大模型 API 自动按《丹方》提炼 -> 生成完整的离线上下文丹。

价值：彻底打破聊天框物理限制，实现“无感知无缝换机”。

💡 一句话总结：从“手动凭感觉防御”升级为“基于本地化数据 + 自动化脚本的兜底方案”。

## 2.2.优化
目前的搜索智能体存在两个问题:
+ 官方默认的tavily工具过于复杂
+ 结果中不包含网页数据源,可信度低

解决思路:
+ 自定义tavily工具
+ 结构化输出
### 自定义tavily工具
LangChain官方提供的tavily工具包含了完整的参数列表,会导致额外的流量和Token消耗。因此,对于简单的业务,我们建议大家利用tavily自定义工具。

In [49]:
from langchain_tavily import TavilySearch
from langchain.tools import tool

# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)
# 然后自己封装为tool
@tool
def web_search(query:str):
    """Search the web for information"""
    return tavily.invoke(query)

# =========================================DeepSeek的方法就加了两包。。。
# from langchain_tavily import TavilySearch
# from langchain.tools import tool

# # 把前面的 tool 变量改个名，比如叫 my_tavily
# my_tavily = TavilySearch(max_results=5, topic="general")

# # 这里的 @tool 就不会被遮蔽了
# @tool
# def web_search(query: str) -> str:
#     """Search the web for information"""
#     return my_tavily.invoke(query)

### 定义结构化输出实体

In [50]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer foruser")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [51]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题",
    response_format=AnswerInfo
)

# # ================DeepSeek的调用模板================DeepSeek的调用模板================DeepSeek的调用模板
# # 2. 手动创建 DeepSeek 模型实例
# deepseek_model = init_chat_model(
#     model="deepseek-chat",              # ① 具体型号（必须全小写）
#     # model_provider="openai",          # ② 无需伪装，DeepSeek 是 LangChain 的亲儿子
#     # base_url=os.getenv("DASHSCOPE_BASE_URL"), # ③ 这是千问的地址，必须注释掉
#     api_key=os.getenv("DEEPSEEK_API_KEY") # ④ 专属钥匙（确保 .env 里是全大写）
# )

# # 3. 创建智能体（注意：这里传的是上面创建好的对象 deepseek_model）
# agent = create_agent(
#     model=deepseek_model, # 必须传对象，不是传字符串！
#     tools=[web_search],
#     system_prompt="你是一个智能助手，你使用工具来解决用户问题",
#     response_format=AnswerInfo
# )
# # ================DeepSeek的调用模板================DeepSeek的调用模板================DeepSeek的调用模板

# # 4. 调用
# for token, metadata in agent.stream(
#     {"messages": [HumanMessage(content="蒸蚌是什么梗??")]},
#     stream_mode="messages"
# ):
#     print(token.content, end="", flush=True)


# # =============================================================DeepSeek端上来的好看的排版
# from IPython.display import display, Markdown

# # 1. 拿到结构化对象
# structured = response["structured_response"]

# # 2. 动手组装：把回答正文和参考文献拼接成一份完整的 Markdown 文本
# final_md = structured.answer + "\n\n---\n\n### 📚 参考来源\n"

# # 遍历所有的 Reference 对象，变成可点击的超链接
# for i, ref in enumerate(structured.reference, 1):
#     final_md += f"{i}. [{ref.title}]({ref.url})\n"

# # 3. 让 Jupyter 直接渲染成漂亮的网页排版
# display(Markdown(final_md))
# # =============================================================DeepSeek端上来的好看的排版

structured = response["structured_response"]

# 直接打印纯文本，不管格式，能看就行
print(structured.answer)

# 纯文本打印引用列表
print("\n参考链接：")
for ref in structured.reference:
    print(f"- {ref.title}: {ref.url}")

「蒸蚌」是「真棒」的谐音梗，本身没有任何额外含义，就是"真棒"的一种可爱、魔性的读法。

## 梗的来源
这个梗出自抖音博主 **@超级无敌大开门** 训练自家三花猫「大开门」分辨「胡萝卜／纸巾」的视频：
- 主人把萝卜、纸巾等道具摆出来让猫选，猫搞不清规则，全程靠「察言观色」，偷偷瞄主人表情、犹豫不决地伸爪子；
- 无论对错，主人都会用一段极具感染力的（湖北口音、米老鼠式破音）高音喊一声 **「真棒！！」** 作为鼓励。

视频被搬到微博后，评论区把主人那句「真棒」**空耳**成了「蒸蚌」，再配上猫咪一脸懵圈的截图，魔性谐音梗就此成型，随后衍生出大量「萝卜纸巾大笨猫」表情包，成了年底全网热梗。

## 为什么火
1. **低门槛＋强反差**：任何人都能对宠物、朋友喊一句「蒸蚌」，瞬间 get「猫咪蒙题」的喜感。猫的「笨」其实是一种允许试错、松弛的可爱，配上「二选一，蒙一个，万一对了呢」的随缘哲学，精准戳中当代年轻人害怕犯错、渴望被肯定的精神状态。
2. **全民共创**：框架极简，衍生出「小猫考研版」「文旅版」「狗子版」「真人版」「打工人版」等各种二创。
3. **明星跟风**：周深在品牌直播中被主持人用「萝卜纸巾猫」同款色卡考问，0.01 秒答对后顺势喊了声「蒸蚌」，现场爆笑，工作室还做了「蒸蚌」签名小卡当周边，完成"猫传人 → 明星传粉"的连锁反应。
4. **商业联动**：这只猫「大开门」甚至拿下了品牌全球代言，文旅、品牌纷纷跟进玩梗。

## 小提醒
- 「蒸蚌」目前**没有额外含义**，就是「真棒」的可爱读法，可以放心用；
- 对宠物模仿玩梗时，记得「只拍照别真考」，避免让猫狗过度训练。
- 另外，B 站等平台也有更早的「蒸蚌」谐音用法，但 2025 年底这波出圈，主要是靠「萝卜纸巾猫」带火的。

参考链接：
- 周深 蒸蚌（新浪新闻）: https://www.sina.cn/news/detail/5247814261015270.html
- 萝卜纸巾大笨猫表情包 | 蒸蚌！（知乎）: https://zhuanlan.zhihu.com/p/1987545001018807313
- 萝卜纸巾分不清的"蒸蚌小猫"，拿下首个全球代言！（广告门）: https://www.adquan.com/article/359318
- 萝卜纸巾蒸蚌！全网小猫